In [1]:
import pandas as pd
from transformers import AutoTokenizer
from transformers import AutoTokenizer
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
import torch
import torch.nn as nn
import math
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
import os
import tqdm 

/Users/maryamsaad/Documents/arabic_poetry_generator/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

df=pd.read_csv("/Users/maryamsaad/Documents/arabic_poetry_generator/datasets/cleaned_poetry_dataset.csv",encoding='utf-8')

In [3]:
df[:2]

,Unnamed: 0,العصر,البيت,الشاعر,clean_text,word_count,word_count_bin
0,94,الحديث,هَمَّتِ الفُلكُ وَاِحتَواها الماءُ وَحَداها...,أحمد شوقي,همت الفلك واحتواها الما وحداها بمن تقل الرجا,8,"(5, 10]"
1,95,الحديث,ضَرَبَ البَحرُ ذو العُبابِ حَوالَي ها سَماء...,أحمد شوقي,ضرب البحر ذو العباب حوالي ها سما قد اكبرتها السما,10,"(5, 10]"


## Byte pair tokenization

In [4]:
class BPETokenizer:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained("konstantindobler/mistral7b-ar-tokenizer-swap-pure-bf16")
    
    def encode(self, text):
        return self.tokenizer.encode(text, add_special_tokens=True)
    
    def get_token_frequencies(self, texts):
        token_freqs = defaultdict(int)
        for text in texts:
            tokens = self.encode(text)
            for token in tokens:
                token_freqs[token] += 1
        return token_freqs
    


In [5]:
# Example usage:
df = pd.read_csv("/Users/maryamsaad/Documents/arabic_poetry_generator/datasets/cleaned_poetry_dataset.csv", encoding='utf-8')
bpe_tokenizer = BPETokenizer("konstantindobler/mistral7b-ar-tokenizer-swap-pure-bf16")
texts = df['clean_text'].dropna().astype(str)
encoded_texts = texts.apply(bpe_tokenizer.encode)
token_freqs = bpe_tokenizer.get_token_frequencies(texts)
vocab_size = bpe_tokenizer.tokenizer.vocab_size  # Always use the tokenizer's vocab_size
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 32768


## Build model

In [6]:
batch_size = 16

In [7]:

train_texts, val_texts = train_test_split(df['clean_text'].dropna().astype(str), test_size=0.1, random_state=42)

In [8]:
def load_data(encoded_dataset, batch_size=16):
    dataset = TokenDataset(encoded_dataset)
    dataloader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn 
    )
    return dataloader
def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    padded = pad_sequence(input_ids, batch_first=True, padding_value=0)
    return {'input_ids': padded}

class TokenDataset(torch.utils.data.Dataset):
    def __init__(self, token_lists):
        self.token_lists = token_lists
    def __len__(self):
        return len(self.token_lists)
    def __getitem__(self, idx):
        tokens = self.token_lists[idx]
        return {'input_ids': torch.tensor(tokens, dtype=torch.long)}
 

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [10]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dim_ff, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.layers(x)


In [11]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_ff, dropout):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dim_ff, dropout)


    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Self-Attention block
        h = self.ln1(x)
        attn_out, _ = self.self_attn(
            h, h, h,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask
        )
        x = x + attn_out

        # Feed-Forward block
        h = self.ln2(x)
        x = x + self.ff(h)

        return x



In [12]:
class TP_PoetDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderBlock(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attn_mask=None, padding_mask=None):
        x = self.token_embedding(input_ids) * math.sqrt(self.token_embedding.embedding_dim)
        x = self.positional_encoding(x)
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask, key_padding_mask=padding_mask)
        return self.lm_head(self.final_norm(x))


In [13]:
vocab_size = bpe_tokenizer.tokenizer.vocab_size
print("Vocab size:", vocab_size)

Vocab size: 32768


In [14]:
model = TP_PoetDecoder(
    vocab_size=vocab_size
)

In [15]:
train_encoded = train_texts.apply(bpe_tokenizer.encode).tolist()
train_dataloader = load_data(train_encoded, batch_size=batch_size)

In [16]:
def train_function(model, dataloader, num_epochs, vocab_size):
    checkpoint_dir='checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.to(device='mps')
    model.train()
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    for epoch in tqdm.tqdm(range(num_epochs)):
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to('mps') 
            inputs = input_ids[:, :-1]
            targets = input_ids[:, 1:]
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

        checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch+1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_loss
        }, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")


In [ ]:
train_function(model, train_dataloader, num_epochs=12, vocab_size=vocab_size)

  0%|          | 0/12 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1 completed. Average Loss: 4.9406


  8%|▊         | 1/12 [36:21<6:39:58, 2181.65s/it]

Checkpoint saved: checkpoints/model_epoch_1.pt
Epoch 2 completed. Average Loss: 2.6134


 17%|█▋        | 2/12 [1:16:53<6:28:06, 2328.63s/it]

...

Checkpoint saved: checkpoints/model_epoch_3.pt
Epoch 4 completed. Average Loss: 0.6750


 33%|███▎      | 4/12 [3:30:36<7:56:39, 3574.97s/it]

Checkpoint saved: checkpoints/model_epoch_4.pt
Epoch 5 completed. Average Loss: 0.4154


 42%|████▏     | 5/12 [4:13:16<6:14:23, 3209.07s/it]

Checkpoint saved: checkpoints/model_epoch_5.pt


In [ ]:
..

In [ ]:
train_dataloader = load_data(train_encoded, batch_size=batch_size)
val_texts = val_texts.apply(bpe_tokenizer.tokenize).tolist()
val_dataloader = load_data(val_texts, batch_size=batch_size)
def test_model(model,dataloader_dataset):
    with torch.no_grad():
        model.eval()
        total_loss = 0
        criterion = torch.nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding
        for batch_idx, batch in enumerate(dataloader_dataset):
            input_ids = batch['input_ids']  # shape: (batch_size, seq_len)
            
            # Shift targets: predict next token
            inputs = input_ids[:, :-1]  # All tokens except last
            targets = input_ids[:, 1:]  # All tokens except first
            
            logits = model(inputs)  # shape: (batch_size, seq_len-1, vocab_size)
            
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
            total_loss += loss.item()
            loss_value = loss.item()
            print(f"Batch {batch_idx+1} Loss: {loss_value:.4f}")
        
        avg_loss = total_loss / len(dataloader_dataset)
        print(f"Test Average Loss: {avg_loss:.4f}")

test_model(model,val_dataloader)    


Batch 1 Loss: 1.0422
Batch 2 Loss: 0.6916
Batch 3 Loss: 0.8282
Batch 4 Loss: 1.0242
Batch 5 Loss: 1.0109
Batch 6 Loss: 0.8164
Batch 7 Loss: 1.0536
Batch 8 Loss: 1.1960
Batch 9 Loss: 0.9192
Batch 10 Loss: 0.9893
Batch 11 Loss: 1.0402
Batch 12 Loss: 1.1824
Batch 13 Loss: 0.6384
Batch 14 Loss: 1.0794
Batch 15 Loss: 1.1740
Batch 16 Loss: 0.7553
Batch 17 Loss: 0.9956
Batch 18 Loss: 1.0165
Batch 19 Loss: 0.9255
Batch 20 Loss: 0.6720
Batch 21 Loss: 0.9541
Batch 22 Loss: 0.8735
Batch 23 Loss: 0.8633
Batch 24 Loss: 0.7542
Batch 25 Loss: 0.8091
Batch 26 Loss: 0.8028
Batch 27 Loss: 0.7374
Batch 28 Loss: 1.0070
Batch 29 Loss: 0.9070
Batch 30 Loss: 0.6238
Batch 31 Loss: 0.6780
Batch 32 Loss: 0.9039
Batch 33 Loss: 0.5914
Batch 34 Loss: 0.7350
Batch 35 Loss: 0.9004
Batch 36 Loss: 1.0359
Batch 37 Loss: 1.0940
Batch 38 Loss: 0.9868
Batch 39 Loss: 0.7763
Batch 40 Loss: 0.9617
Batch 41 Loss: 1.0218
Batch 42 Loss: 0.6834
Batch 43 Loss: 0.9156
Batch 44 Loss: 0.9820
Batch 45 Loss: 1.0003
Batch 46 Loss: 0.70